# Customer behaviour analysis project — EDA

The project aims to answer: **How the company can leverage consumer shopping data to identify trends, improve consumer engagement, and optimize marketing and product strategies?**

## 1. Importing libraries and loading the dataset 

In [53]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("./data/customer_shopping_behavior.csv")

## 2. Data overview & sanity checks.

In [28]:
df.head() #viewing the first five columns of the dataset

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [7]:
df.tail() #viewing the last five rows of the dataset

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,4.2,No,2-Day Shipping,No,No,32,Venmo,Weekly
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,4.5,No,Store Pickup,No,No,41,Bank Transfer,Bi-Weekly
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,2.9,No,Standard,No,No,24,Venmo,Quarterly
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,3.8,No,Express,No,No,24,Venmo,Weekly
3899,3900,52,Female,Handbag,Accessories,81,California,M,Beige,Spring,3.1,No,Store Pickup,No,No,33,Venmo,Quarterly


In [29]:
df.info() #Checking the completeness of the data, available data types, rows and column names.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [54]:
#Null values per column
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

The [**Review Rating**] column seems to be missing a few observations, and this could be as a reuslt of many different cases. Nonetheless, the way forward is to fill the missing values with the median.

Best practice is to fill the data with the median of the dataset because the median is usually unaffected by outliers and won't introduce inflated skewedness to our data

In [55]:
#Addressing the null values -> Grouping by category then using the median from each category to fill in the missing data in the corresponding category
df["Review Rating"] = df.groupby("Category")["Review Rating"].transform(lambda x: x.fillna(x.median()))

Form stronger understanding of the "Transform line including the lambda function" in this instance.

In [56]:
#Rechecking the null values to confirm whether the code was successful

df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [33]:
#Summary statistics of the dataset

df.describe().T

,count,mean,std,min,25%,50%,75%,max
Customer ID,3900.0,1950.500000,1125.977353,1.0,975.75,1950.5,2925.25,3900.0
Age,3900.0,44.068462,15.207589,18.0,31.00,44.0,57.00,70.0
Purchase Amount (USD),3900.0,59.764359,23.685392,20.0,39.00,60.0,81.00,100.0
Previous Purchases,3900.0,25.351538,14.447125,1.0,13.00,25.0,38.00,50.0


In [57]:
#Updating column names by enforcing the snake casing convention

df.columns = df.columns.str.lower() #turn the upper cases to lower case
df.columns = df.columns.str.replace(" ", "_") #replace spaces inbetween words with underscores
df = df.rename(columns={'purchase_amount_(usd)':'purchase_amount'}) #ensuring consistency in column naming 

print(df.columns) #print results

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')


In [58]:
#Creating a column for age_group -> specify the age group where each age falls under.
labels = ["Young adult", "Adult", "Middle-aged", "Senior"]
df["age_group"] = pd.qcut(df["age"], q=4, labels=labels) #qcut splits the data, q specifies how many times we need to split it (number is based on the labels we provided)

In [12]:
#Checking if the code worked.
df[["age", "age_group"]].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young adult
2,50,Middle-aged
3,21,Young adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young adult
8,26,Young adult
9,57,Middle-aged


In [42]:
df["frequency_of_purchases"].unique()

array(['Fortnightly', 'Weekly', 'Annually', 'Quarterly', 'Bi-Weekly',
       'Monthly', 'Every 3 Months'], dtype=object)

In [59]:
#Creating column for purchase frequency days -> converting the frequencies to numerical values since textual data can be harder to analyse

frequency_mapping = {
    "Fortnightly": 14,
    "Weekly": 7,
    "Annually": 365,
    "Quarterly": 90,
    "Bi-Weekly": 14,
    "Monthly": 30,
    "Every 3 Months": 90}

df["purchase_frequency"] = df["frequency_of_purchases"].map(frequency_mapping)

In [14]:
df[["frequency_of_purchases","purchase_frequency"]].head(8)

,frequency_of_purchases,purchase_frequency
0,Fortnightly,14
1,Fortnightly,14
2,Weekly,7
3,Weekly,7
4,Annually,365
5,Weekly,7
6,Quarterly,90
7,Weekly,7


In [74]:
df[["discount_applied", "promo_code_used"]].head()

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes


In [15]:
#Are every values the same for both columns? (Aim is to reduce redundancy)
(df["discount_applied"] == df["promo_code_used"]).all()

np.True_

In [60]:
#Dropping Promo code used column

df = df.drop("promo_code_used", axis=1)

df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency'],
      dtype='object')

In [1]:
pip install pyodbc

Note: you may need to restart the kernel to use updated packages.


In [63]:
#Checking the available drivers to connect to the SQL Server
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [ ]:
#Connecting the notebook to SQL Server Management Studios (SSMS) to conduct further analysis using SQL
from sqlalchemy import create_engine
from urllib.parse import quote_plus

#Connecting to the database
params = quote_plus("DRIVER={ODBC Driver 17 for SQL Server};"
                   "SERVER=FRANKSOCEAN\\SQLEXPRESS;"
                   "DATABASE=Portfolio Project;"
                   "Trusted_Connection=yes;"
                   "TrustServerCertificate=yes;")
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

#Writing the dataframe into the database
df.to_sql("customers", engine, if_exists="replace", index=False)

In [62]:
#Reading back the data sample to ensure successful confi
pd.read_sql("SELECT TOP 5 * FROM customers;", engine)

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,365
